<a href="https://colab.research.google.com/github/AndreySeleznov/EveryoneCanCode-US/blob/main/RAG_HR_with_API_KEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai PyPDF2 gradio

In [ ]:
import google.generativeai as genai
import os
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader

In [ ]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)
print("google.generativeai library imported and configured.")

## Етап 1: зчитування даних

In [ ]:
df = pd.DataFrame(columns=['Title','Text'])
df

In [ ]:
for file_name in os.listdir('.'):
    if file_name.endswith('.pdf'):
        try:
            with open(file_name,'rb') as file:
                pdf_reader = PdfReader(file)
                text = ''
                for page_num in range(len(pdf_reader.pages)):
                    text+=pdf_reader.pages[page_num].extract_text()
                    text = text.replace('\n', ' ')
                new_row = pd.DataFrame({'Title':[file_name], 'Text':[text]})
                df = pd.concat([df,new_row],ignore_index=True)
        except Exception as e:
            print(f"Error processing file {file_name}: {e}")

In [ ]:
df

In [ ]:
print(df['Text'].iloc[1])

## Етап 2: Векторизація

In [ ]:
texts = df['Text']
task_type = 'RETRIEVAL_DOCUMENT'
result = genai.embed_content(model="gemini-embedding-001", content=texts)
print(result)


In [ ]:
print(result['embedding'])

In [ ]:
def embed_text(text: str) -> np.ndarray:
    resp = genai.embed_content(
        model="gemini-embedding-001",
        content=text,
        task_type=task_type,   # "retrieval_query" or "retrieval_document"
    )
    return np.asarray(resp["embedding"], dtype=np.float32)

In [ ]:
df['Embeddings'] = df['Text'].apply(embed_text)
df

In [ ]:
def query_similarity_score(query,vector):
    # query --> 'How can I take a sick leave?' TEXT STR
    # vector --> our existing Array (list)

    # Query STR --> LIST
    query_embedding = embed_text(query)

    #SIMILARITY
    return np.dot(query_embedding,vector)

In [ ]:
query = "Скільки в мене на рік доступно вихідних?"
df['Similarity'] = df['Embeddings'].apply(lambda vector: query_similarity_score(query,vector))

In [ ]:
df

In [ ]:
#now let`s extract `the text we need to have as context to answer our question
df.sort_values('Similarity',ascending=False)
df.sort_values('Similarity',ascending=False)[['Title','Text']]
df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]

In [ ]:
def  most_similar_doc(query):
    df['Similarity'] = df['Embeddings'].apply(lambda vector: query_similarity_score(query,vector))
    title = df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]['Title']
    text = df.sort_values('Similarity',ascending=False)[['Title','Text']].iloc[0]['Text']
    return title,text

In [ ]:
title, text = most_similar_doc('Хочу відвідати конференцію. Чи може компанія мені це покрити?')
print(title,text)

## Етап 3: Генерація відповіді

In [ ]:
def rag_implement(query):
    #Retrival Document
    title,text = most_similar_doc(query)
    #AG - Model
    model = genai.GenerativeModel('gemini-2.5-flash')
    #Augment --> prompt engineering
    prompt = f'Answer this query:\n{query}. \n Only use this context to answer: \n{text}'
    #Sourcing(title) - what doc was used
    response = model.generate_content(prompt)
    # return f'{response.text}\n\n Source Doc Title: {title}'
    return response.text,title

In [ ]:
print(rag_implement('А чи є курси французької?'))

## Chunking

In [ ]:
#Simple chunking
def text_splitter(text, chunk_size=500, overlap=50):
    chunks = []
    # Ensure text is a string
    if not isinstance(text, str):
        text = str(text)

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

In [ ]:
# Create a list to store the chunked data
chunked_data = []

for index, row in df.iterrows():
    title = row['Title']
    full_text = row['Text']

    # Split the full text into chunks
    chunks = text_splitter(full_text)

    for chunk in chunks:
        chunk_embedding = embed_text(chunk)
        chunked_data.append({'Title': title, 'Text': chunk, 'Embeddings': chunk_embedding})

In [ ]:
# Create a new DataFrame from the chunked data
df_chunks = pd.DataFrame(chunked_data)

display(df_chunks.head())

In [ ]:
df_chunks = df_chunks.reset_index(drop=True)

df_chunks["Embeddings"] = df_chunks["Embeddings"].apply(lambda x: np.asarray(x, dtype=np.float32))

EMB_MATRIX = np.vstack(df_chunks["Embeddings"].to_numpy())
EMB_MATRIX_N = EMB_MATRIX / (np.linalg.norm(EMB_MATRIX, axis=1, keepdims=True) + 1e-12)

Get the most relevant chunks to the query

In [ ]:
def get_relevant_chunks(query: str, num_chunks: int = 3):
    q = np.asarray(embed_text(query), dtype=np.float32)   # one API call
    sims = EMB_MATRIX @ q                                 # dot product vs all chunks
    top_idx = np.argsort(-sims)[:num_chunks]
    top = df_chunks.iloc[top_idx]
    return top["Text"].tolist(), top["Title"].tolist()

In [ ]:
get_relevant_chunks("Скільки в мене на рік доступно вихідних?")

## Improving chunking - by split per sentence + adding preprocessing

In [ ]:
import nltk
import re
from nltk.tokenize import sent_tokenize

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
def text_splitter(text, chunk_size=500, overlap=50):
    chunks = []
    # Ensure text is a string
    if not isinstance(text, str):
        text = str(text)

    # Preprocessing: remove extra whitespace and unnecessary symbols
    # Replace multiple spaces/newlines with a single space.
    text = re.sub(r'[\s]+', ' ', text)
    # Keep alphanumeric characters, spaces, and common punctuation.
    # Added Cyrillic character range (а-яА-Я) for Ukrainian text.
    text = re.sub(r'[^a-zA-Zа-яА-ЯІіЇїЄє0-9\s.,!?;:]', '', text)
    text = text.strip()

    sentences = sent_tokenize(text)

    current_sentence_index = 0
    while current_sentence_index < len(sentences):
        current_chunk_sentences = []
        current_chunk_length = 0

        # Add sentences from the end of the previous chunk for overlap
        if len(chunks) > 0:
            last_chunk_sentences = sent_tokenize(chunks[-1])
            temp_overlap_chars = 0
            overlap_sentences_for_this_chunk = []

            # Iterate backwards in the previous chunk's sentences to find enough for overlap
            for i in range(len(last_chunk_sentences) - 1, -1, -1):
                sentence = last_chunk_sentences[i]
                # Add 1 for the space that will be between sentences when joined
                if temp_overlap_chars + len(sentence) + (1 if temp_overlap_chars > 0 else 0) <= overlap:
                    overlap_sentences_for_this_chunk.insert(0, sentence) # Add to front to maintain order
                    temp_overlap_chars += len(sentence) + (1 if temp_overlap_chars > 0 else 0)
                else:
                    break
            current_chunk_sentences.extend(overlap_sentences_for_this_chunk)
            current_chunk_length = len(" ".join(current_chunk_sentences)) if current_chunk_sentences else 0

        # Now add new sentences from the `sentences` list until chunk_size is met
        temp_sentence_idx = current_sentence_index
        while temp_sentence_idx < len(sentences):
            sentence = sentences[temp_sentence_idx]
            # Check if adding this sentence will exceed chunk_size
            # Add 1 for the space that will be between sentences when joined
            if current_chunk_length + len(sentence) + (1 if current_chunk_sentences else 0) <= chunk_size:
                current_chunk_sentences.append(sentence)
                current_chunk_length += len(sentence) + (1 if current_chunk_sentences else 0)
                temp_sentence_idx += 1
            else:
                break # Current chunk is full enough

        # Handle cases where a single sentence is larger than chunk_size, or if no sentences were added
        if not current_chunk_sentences and current_sentence_index < len(sentences):
            # If the current_chunk_sentences is empty, it means the very first sentence is too long or no sentences were processed yet
            # Take the current sentence, even if it's too long, as a single chunk
            current_chunk_sentences.append(sentences[current_sentence_index])
            temp_sentence_idx = current_sentence_index + 1 # Advance pointer

        if current_chunk_sentences:
            chunks.append(" ".join(current_chunk_sentences))

        current_sentence_index = temp_sentence_idx # Move to the point where the next chunk should start

    return chunks

In [ ]:
chunked_data = []

for index, row in df.iterrows():
    title = row['Title']
    full_text = row['Text']

    # Split the full text into chunks
    chunks = text_splitter(full_text)

    # For each chunk, create a new entry with title, chunk text, and embedding
    for chunk in chunks:
        chunk_embedding = embed_text(chunk)
        chunked_data.append({'Title': title, 'Text': chunk, 'Embeddings': chunk_embedding})

# Create a new DataFrame from the chunked data
df_chunks = pd.DataFrame(chunked_data)

display(df_chunks.head())

In [ ]:
df_chunks = df_chunks.reset_index(drop=True)

df_chunks["Embeddings"] = df_chunks["Embeddings"].apply(lambda x: np.asarray(x, dtype=np.float32))

EMB_MATRIX = np.vstack(df_chunks["Embeddings"].to_numpy())
EMB_MATRIX_N = EMB_MATRIX / (np.linalg.norm(EMB_MATRIX, axis=1, keepdims=True) + 1e-12)

In [ ]:
get_relevant_chunks("Скільки в мене на рік доступно вихідних?")

In [ ]:
def rag_implement_chunked(query):
    # Retrieval of relevant chunks
    relevant_texts, source_titles = get_relevant_chunks(query, num_chunks=3)

    # Concatenate relevant texts for the prompt
    context = " ".join(relevant_texts)

    # AG - Model
    model = genai.GenerativeModel('gemini-2.5-flash')

    # Augment --> prompt engineering
    prompt = f'Answer this query:\n{query}. \n Only use this context to answer: \n{context}'

    response = model.generate_content(prompt)

    # Sourcing (titles of documents from which chunks were taken)
    # Get unique titles from the documents that contributed the relevant chunks
    source_titles = df_chunks[df_chunks['Text'].isin(relevant_texts)]['Title'].unique().tolist()

    return response.text, "\n".join(sorted(set(source_titles)))

In [ ]:
rag_implement_chunked("Скільки в мене на рік доступно вихідних?")

## Extending the user query

In [ ]:
def extend_query(original_query):
    model = genai.GenerativeModel('gemini-2.5-flash')
    prompt = f"""Given the query: '{original_query}', generate 2-3 alternative ways to phrase this question that could be used to find similar information.
    Return each variant on a new line and include the original query as one of the options.
    Do not include any commentary, introductory or concluding text, just the list of queries.
    The result must be in the language of the original query, no translations. If it`s in English, return variants in English, if in Ukrainian, return in Ukrainian, etc."""
    response = model.generate_content(prompt)
    extended_queries = [q.strip() for q in response.text.split('\n') if q.strip()]
    if original_query not in extended_queries:
        extended_queries.insert(0, original_query) # Ensure original query is always included
    return extended_queries

In [ ]:
extend_query("Чи є курси французької?")

In [ ]:
def rag_implement_chunked_extended(query):
    # Extend the query
    extended_queries = extend_query(query)

    # Retrieve relevant chunks for each extended query
    all_relevant_texts = []
    for ext_query in extended_queries:
        all_relevant_texts.extend(get_relevant_chunks(ext_query, num_chunks=2)) # Get 1 chunk per extended query
    combined_context_chunks = []
    for ext_query in extended_queries:
        combined_context_chunks.extend(get_relevant_chunks(ext_query, num_chunks=1))

    # Remove duplicates while preserving order
    # Convert to set and back to list, or iterate and add if not already present
    seen_chunks = set()
    unique_combined_context_chunks = []
    for chunk_text in combined_context_chunks:
        if chunk_text not in seen_chunks:
            unique_combined_context_chunks.append(chunk_text)
            seen_chunks.add(chunk_text)

    # Limit the number of chunks passed to the LLM to avoid token limits
    final_relevant_texts = unique_combined_context_chunks[:3] # Using up to 3 unique chunks

    context = " ".join(final_relevant_texts)

    model = genai.GenerativeModel('gemini-2.5-flash')
    prompt = f'Answer this query:\n{query}. \n Only use this context to answer: \n{context}'

    response = model.generate_content(prompt)

    source_titles = []
    for chunk_text in final_relevant_texts:
        titles_for_chunk = df_chunks[df_chunks['Text'] == chunk_text]['Title'].unique().tolist()
        source_titles.extend(titles_for_chunk)
    source_titles = list(set(source_titles)) # Get unique titles

    return response.text, source_titles

In [ ]:
def rag_implement_chunked_extended(query: str):
    extended_queries = extend_query(query)

    combined_context_chunks = []
    source_titles = []

    # get 1 top chunk per extended query
    for ext_query in extended_queries:
        texts, titles = get_relevant_chunks(ext_query, num_chunks=1)  # unpack
        combined_context_chunks.extend(texts)
        source_titles.extend(titles)

    # remove duplicate chunks while preserving order
    seen = set()
    unique_chunks = []
    for t in combined_context_chunks:
        if t not in seen:
            unique_chunks.append(t)
            seen.add(t)

    final_relevant_texts = unique_chunks[:3]
    context = " ".join(final_relevant_texts)

    model = genai.GenerativeModel("gemini-2.5-flash")
    prompt = f"Answer this query:\n{query}\nOnly use this context to answer:\n{context}"
    response = model.generate_content(prompt)

    used_titles = df_chunks[df_chunks["Text"].isin(final_relevant_texts)]["Title"].unique().tolist()

    return response.text, "\n".join(sorted(set(used_titles)))

In [ ]:
rag_implement_chunked_extended("Чи є курси французької?")

In [ ]:
import gradio as gr

description = "Welcome to HR policies bureau. Here you can ask anything about onboarding process, medical insuranse, leave policies and benefits."
title = "HR Wiki 👾"
output1 = gr.Textbox(label="Answer")
output2 = gr.Textbox(label="Where it comes from?")
gr.Interface(fn=rag_implement_chunked,
             title=title,
             description=description,
             inputs=["text"],
             outputs=[output1,output2]).launch(debug=True)